# Module 8 – Lab 2: Nomination Management Agent with Excel Integration

**Advanced LLM Systems · LangChain custom tools · Pydantic validation · Excel as a business data source**

In this lab you build a conversational assistant for employee nominations. The agent reads a real Excel workbook, collects nomination details through conversation, validates them with Pydantic, searches records by employee name or employee ID, answers questions from the retrieved data, and adds new nominations to the workbook **only after you confirm them**.

> This is a training application, not a production HR system. All employees, IDs and records are fictional.

## Learning objectives

By the end of this lab you will be able to:

1. Explain custom tools in LangChain and how an agent invokes them.
2. Read and inspect an existing Excel workbook with Pandas.
3. Validate structured data with a lightweight Pydantic model.
4. Write simple Python functions that search and update Excel.
5. Wrap those functions as LangChain tools with `@tool`.
6. Build a tool-calling agent that collects information through conversation.
7. Require explicit confirmation before any write, and detect potential duplicates.
8. Show how an agent combines conversation, tool calling and structured business data.

## SECTION 1 – Introduction to Custom Business Agents

**What is an AI agent?** An agent is an LLM that decides what to do next. Instead of answering only from memory, it can call *tools* (Python functions), read their results, and continue until it can give an answer.

**What is a tool?** A tool is a function the LLM is allowed to call. A *built-in* tool comes ready-made with a framework, such as a web search. A *custom business tool* is a function we write for our own process, such as "search nominations by employee ID" or "add a nomination to the workbook".

**What is tool calling?** The LLM does not run code itself. It returns a structured request such as "call `search_nomination_by_employee_id` with `employee_id = EMP1001`". LangChain runs the Python function, sends the output back to the LLM, and the LLM writes the final answer.

**How an agent invokes a custom tool**

1. The user sends a message.
2. The LLM reads the message and the tool descriptions, then picks a tool (or answers directly).
3. LangChain runs the Python function and returns its output to the LLM.
4. The LLM uses that output to answer. If a write is needed, it asks the user to confirm first.

**How business applications connect to Excel.** The workbook is the business data source. Python reads it with Pandas, and writes new rows with OpenPyXL. No database is involved.

**Why structured data validation matters.** Text typed in a conversation is messy. A Pydantic model checks every field (a valid employee ID, a real date, a non-blank reason) before anything reaches Excel. Bad data is rejected instead of saved.

### Architecture of this lab

```mermaid
flowchart LR
    A[User Conversation] --> B[LangChain Agent]
    B --> C[Custom Tools]
    C --> D[Excel Workbook]
    D --> E[Retrieved Data]
    E --> F[Agent Response]
```

Each step in the diagram is built and tested in this notebook. If the diagram shows as plain text in your editor, that is only a display setting. The flow is the same.

| | Module 8 Lab 1 | Module 8 Lab 2 (this lab) |
|---|---|---|
| Tools | Existing (built-in) tools: File Management, Python REPL, web search | Custom tools **we write** for nomination management |
| Data | Files in an operations workspace | The `employee_nominations.xlsx` business workbook |
| Safety | Agent limited to one workspace folder | Pydantic validation and explicit confirmation before every write |
| Typical question | "Read the log and summarise it." | "Has Rahul Sharma been nominated before?" |

## SECTION 2 – Environment Setup

This lab runs on the **Python interpreter already configured in VS Code**. Do not create a new virtual environment, install another Python, or change the default interpreter. Install the dependencies into that same interpreter with `python -m pip install -r requirements.txt` (see the README).

The cell below:

1. imports the libraries,
2. shows the Python version and the installed LangChain versions,
3. loads the `.env` file and checks that an OpenAI key is available (the key itself is never printed),
4. sets the model configuration in one place.

In [ ]:
# ---- Standard library and data helpers ----
import os
import re
import shutil
import sys
from datetime import date
from difflib import SequenceMatcher
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Annotated, Optional

import pandas as pd
from dotenv import load_dotenv
from openpyxl import load_workbook
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

# ---- LangChain: messages, prompts, tools, model and agent ----
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

print("Libraries imported.")
print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print()
for package in ["langchain", "langchain-core", "langchain-openai", "langchain-classic",
                "pandas", "openpyxl", "pydantic", "python-dotenv"]:
    try:
        print(f"{package:20} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:20} NOT INSTALLED")

# Read OPENAI_API_KEY from the .env file in this folder (if the file exists)
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY", "")
API_KEY_FOUND = bool(api_key) and api_key != "your_openai_api_key_here"   # the key itself is never printed
print("\nOPENAI_API_KEY available:", API_KEY_FOUND)
if not API_KEY_FOUND:
    print("No key yet: Sections 1 to 10 still run. Section 11 onward (the agent) needs a key in .env.")

# Model settings. Change CHAT_MODEL here to try another OpenAI chat model.
CHAT_MODEL = "gpt-4o-mini"   # same model as Module 7 and Module 8 Lab 1
MAX_ITERATIONS = 8           # safety limit: maximum reasoning and tool steps per request
print("Chat model:", CHAT_MODEL, "| max steps:", MAX_ITERATIONS)

## SECTION 3 – Inspect the Existing Excel Workbook

Excel is a simple, familiar business data source. Each **row** is one nomination record and each **column** is one field. In this lab Excel is the **source of truth**: the notebook reads the workbook whenever it needs data and writes new rows back into it. There is no database.

The file `employee_nominations.xlsx` must sit in the same folder as this notebook. It has one worksheet named `Nominations` with these columns:

| Column | Meaning |
|---|---|
| Nomination_ID | Unique record ID, for example NOM-0001 |
| Employee_ID | Unique employee ID, for example EMP1001 |
| Employee_Name | Employee's full name |
| Department | Employee's department |
| Designation | Employee's job title |
| Award_Category | Award being nominated for |
| Nomination_Reason | Why the employee is nominated |
| Nominated_By | Nominating manager |
| Nomination_Date | Date the nomination was made |
| Status | Pending, Approved or Rejected |

In [ ]:
# Path to the workbook. VS Code runs the notebook from its own folder, so the file is found next to it.
WORKBOOK_PATH = (Path.cwd() / "employee_nominations.xlsx").resolve()
SHEET_NAME = "Nominations"
BACKUP_PATH = WORKBOOK_PATH.with_name(WORKBOOK_PATH.name + ".bak")   # created before the first write

assert WORKBOOK_PATH.exists(), f"Workbook not found at {WORKBOOK_PATH}. Open the notebook from the lab folder."
print("Workbook path:", WORKBOOK_PATH)
print("Exists       :", WORKBOOK_PATH.exists())

# Load the Nominations worksheet with Pandas. IDs are read as text so codes like EMP1001 stay intact.
nominations = pd.read_excel(WORKBOOK_PATH, sheet_name=SHEET_NAME, dtype={"Nomination_ID": str, "Employee_ID": str})
COLUMNS = list(nominations.columns)                      # the expected layout, reused in later checks
ORIGINAL_IDS = nominations["Nomination_ID"].tolist()     # used in Section 14 to prove no record was lost

print("Existing nominations:", len(nominations))
print("Columns:", COLUMNS)
print("\nFirst five records:")
nominations.head(5)

## SECTION 4 – Create a Pydantic Model

**Pydantic** checks data against a model: a class with typed fields. When you create `Nomination(...)`, Pydantic:

- converts types where it can (the text `"2026-04-30"` becomes a real date),
- rejects values that are missing, blank, too short or badly formatted,
- raises a `ValidationError` that lists **every** problem, so the user can correct them.

Our `Nomination` model keeps things simple: eight fields, no nested models, and one small validator that stores employee IDs in upper case. Invalid data is rejected **before** it reaches Excel, because `add_nomination()` only accepts a valid `Nomination` object. The workbook's `Nomination_ID` and `Status` are generated by the application, so they are not part of the model.

Validation rules:

| Field | Rule |
|---|---|
| employee_id | Not blank; format `EMP` followed by 4 digits |
| employee_name | Not blank; 2 to 100 characters |
| department, designation, award_category | Not blank |
| nomination_reason | Not blank; at least 10 characters |
| nominated_by | Not blank |
| nomination_date | A real calendar date (defaults to today) |

In [ ]:
class Nomination(BaseModel):
    """One employee nomination. Every field is checked before anything is written to Excel."""

    model_config = ConfigDict(str_strip_whitespace=True)   # "   " becomes "" and then fails the length check

    employee_id: str = Field(min_length=1, pattern=r"^EMP\d{4}$", description="Format EMP + 4 digits, e.g. EMP1001")
    employee_name: str = Field(min_length=2, max_length=100)
    department: str = Field(min_length=1, max_length=60)
    designation: str = Field(min_length=1, max_length=80)
    award_category: str = Field(min_length=1, max_length=60)
    nomination_reason: str = Field(min_length=10, max_length=500)
    nominated_by: str = Field(min_length=1, max_length=100)
    nomination_date: date = Field(default_factory=date.today)   # must be a real calendar date

    @field_validator("employee_id", mode="before")
    @classmethod
    def store_employee_id_in_upper_case(cls, value):
        # "emp1001" is accepted and stored as "EMP1001"
        return value.strip().upper() if isinstance(value, str) else value


def explain_validation_error(error: ValidationError) -> str:
    """Turn Pydantic's error list into short lines that the user or the agent can act on."""
    return "\n".join(f"- {'.'.join(str(part) for part in err['loc'])}: {err['msg']}" for err in error.errors())


# Demonstration: one valid record and several invalid ones
valid_example = dict(
    employee_id="EMP1001", employee_name="Rahul Sharma", department="IT Operations",
    designation="Senior Systems Engineer", award_category="Star Performer Award",
    nomination_reason="He resolved a critical outage and reduced service downtime.",
    nominated_by="Neha Kapoor, Operations Manager", nomination_date="2026-04-30",
)

test_cases = {
    "valid record": valid_example,
    "blank employee ID": {**valid_example, "employee_id": "   "},
    "missing employee name": {k: v for k, v in valid_example.items() if k != "employee_name"},
    "blank department": {**valid_example, "department": ""},
    "reason too short": {**valid_example, "nomination_reason": "Good"},
    "invalid date (30 Feb)": {**valid_example, "nomination_date": "2026-02-30"},
}

for label, data in test_cases.items():
    try:
        record = Nomination(**data)
        print(f"PASSED   {label}: {record.employee_id}, date {record.nomination_date}")
    except ValidationError as error:
        print(f"REJECTED {label}:\n{explain_validation_error(error)}")

## SECTION 5 – Create Simple Python Functions

Before we wrap anything as an agent tool, we write plain Python functions and test each one on its own. A tool is only a thin layer around a function like these.

| Function | Purpose |
|---|---|
| `read_nominations()` | Read the whole Nominations sheet |
| `search_by_employee_id()` | Rows for one exact employee ID |
| `search_by_employee_name()` | Rows whose name contains the text (case-insensitive) |
| `build_nomination_summary()` | Counts: totals, status, department, award category |
| `add_nomination()` | Append one validated record as **Pending**, saved safely |

The functions are deliberately simple: no classes and no repository pattern. Every read goes back to the Excel file, so the workbook stays the source of truth.

In [ ]:
def read_nominations() -> pd.DataFrame:
    """Read every row of the Nominations worksheet from the Excel file."""
    return pd.read_excel(WORKBOOK_PATH, sheet_name=SHEET_NAME, dtype={"Nomination_ID": str, "Employee_ID": str})


def search_by_employee_id(employee_id: str) -> pd.DataFrame:
    """Return all rows for one employee ID (exact match, case-insensitive). A blank ID matches nothing."""
    df = read_nominations()
    if not employee_id.strip():
        return df.iloc[0:0]
    return df[df["Employee_ID"].str.strip().str.upper() == employee_id.strip().upper()]


def search_by_employee_name(employee_name: str) -> pd.DataFrame:
    """Return rows whose Employee_Name contains the text (partial match, case-insensitive). A blank name matches nothing."""
    df = read_nominations()
    if not employee_name.strip():
        return df.iloc[0:0]
    return df[df["Employee_Name"].str.contains(employee_name.strip(), case=False, regex=False, na=False)]


def build_nomination_summary(group_by: str = "overview") -> pd.Series:
    """Count nominations. 'overview' gives totals and status counts; the other options count one column."""
    df = read_nominations()
    if group_by == "overview":
        status_counts = df["Status"].value_counts()
        return pd.Series({
            "Total nominations": len(df),
            "Pending": int(status_counts.get("Pending", 0)),
            "Approved": int(status_counts.get("Approved", 0)),
            "Rejected": int(status_counts.get("Rejected", 0)),
        })
    columns = {"status": "Status", "department": "Department", "award_category": "Award_Category"}
    if group_by not in columns:
        raise ValueError("group_by must be one of: overview, status, department, award_category")
    return df[columns[group_by]].value_counts()


def format_records(df: pd.DataFrame) -> str:
    """Turn matching rows into readable text for the user or the agent."""
    if df.empty:
        return "No matching records."
    lines = []
    for _, row in df.iterrows():
        lines.append(
            f"- {row['Nomination_ID']} | {row['Employee_ID']} {row['Employee_Name']} | {row['Designation']}, {row['Department']}\n"
            f"  Award: {row['Award_Category']} | Status: {row['Status']} | Date: {str(row['Nomination_Date'])[:10]} | Nominated by: {row['Nominated_By']}\n"
            f"  Reason: {row['Nomination_Reason']}"
        )
    return "\n".join(lines)


# Demonstrate each function on its own
print("1. read_nominations(): rows =", len(read_nominations()))

print("\n2. search_by_employee_id('EMP1001'):")
print(format_records(search_by_employee_id("EMP1001")))

print("\n3. search_by_employee_name('rahul') - partial and case-insensitive:")
print(format_records(search_by_employee_name("rahul")))

print("\n4. build_nomination_summary():")
print(build_nomination_summary().to_string())

print("\n5. build_nomination_summary('department'):")
print(build_nomination_summary("department").to_string())

In [ ]:
# add_nomination() WRITES to the workbook. This demonstration adds one record, Ananya Rao (EMP1015), as Pending.

def find_potential_duplicates(nomination: Nomination) -> pd.DataFrame:
    """Potential duplicate = same employee ID, same award category and a very similar reason (80% or more)."""
    df = read_nominations()
    same_employee_and_award = df[
        (df["Employee_ID"].str.strip().str.upper() == nomination.employee_id)
        & (df["Award_Category"].str.strip().str.lower() == nomination.award_category.lower())
    ]
    if same_employee_and_award.empty:
        return same_employee_and_award
    reason = nomination.nomination_reason.lower()
    is_similar = same_employee_and_award["Nomination_Reason"].apply(
        lambda text: SequenceMatcher(None, str(text).lower(), reason).ratio() >= 0.8
    )
    return same_employee_and_award[is_similar]


def next_nomination_id(df: pd.DataFrame) -> str:
    """Create the next ID, for example NOM-0015, from the highest number already in the sheet."""
    numbers = df["Nomination_ID"].astype(str).str.extract(r"(\d+)")[0].dropna().astype(int)
    return f"NOM-{(int(numbers.max()) if len(numbers) else 0) + 1:04d}"


def save_workbook_safely(new_row: dict) -> None:
    """Append one row to the Nominations sheet without touching the rest of the workbook.

    1. Back up the original workbook once, before the first write.
    2. Append the row with OpenPyXL (other sheets and formatting are kept).
    3. Save to a temporary file and check it.
    4. Replace the original file only when every check passes.
    If anything fails, the original workbook is left as it was.
    """
    if not BACKUP_PATH.exists():
        shutil.copy2(WORKBOOK_PATH, BACKUP_PATH)   # backup before the first write
    before = read_nominations()
    temp_path = WORKBOOK_PATH.with_name("employee_nominations_temp.xlsx")
    try:
        workbook = load_workbook(WORKBOOK_PATH)
        sheet = workbook[SHEET_NAME]
        headers = [cell.value for cell in sheet[1]]
        # Write directly under the last row that holds a Nomination_ID. sheet.max_row can be larger than the
        # data when someone has formatted empty cells in Excel, and appending there would leave a gap.
        id_column = headers.index("Nomination_ID") + 1
        last_row = max([1] + [r for r in range(2, sheet.max_row + 1) if sheet.cell(row=r, column=id_column).value not in (None, "")])
        target_row = last_row + 1
        for column, header in enumerate(headers, start=1):   # values follow the header order
            sheet.cell(row=target_row, column=column).value = new_row.get(header)
        sheet.cell(row=target_row, column=headers.index("Nomination_Date") + 1).number_format = "yyyy-mm-dd"
        workbook.save(temp_path)

        after = pd.read_excel(temp_path, sheet_name=SHEET_NAME, dtype={"Nomination_ID": str, "Employee_ID": str})
        if len(after) != len(before) + 1:
            raise RuntimeError("Check failed: the row count did not grow by exactly one.")
        if list(after.columns) != list(before.columns):
            raise RuntimeError("Check failed: the columns changed.")
        if after["Nomination_ID"].tolist()[:-1] != before["Nomination_ID"].tolist():
            raise RuntimeError("Check failed: existing records changed.")
        os.replace(temp_path, WORKBOOK_PATH)
    except PermissionError as error:
        raise RuntimeError("The workbook is open in another application (for example Excel). Close it and try again.") from error
    finally:
        if temp_path.exists():
            temp_path.unlink()


def add_nomination(nomination: Nomination) -> str:
    """Add one validated nomination with status Pending and return its new Nomination_ID.

    This function does not ask for confirmation or check duplicates. The agent tool does both before calling it.
    """
    nomination_id = next_nomination_id(read_nominations())
    save_workbook_safely({
        "Nomination_ID": nomination_id,
        "Employee_ID": nomination.employee_id,
        "Employee_Name": nomination.employee_name,
        "Department": nomination.department,
        "Designation": nomination.designation,
        "Award_Category": nomination.award_category,
        "Nomination_Reason": nomination.nomination_reason,
        "Nominated_By": nomination.nominated_by,
        "Nomination_Date": nomination.nomination_date,
        "Status": "Pending",   # always Pending: this lab never approves automatically
    })
    return nomination_id


def restore_sample_workbook() -> None:
    """Optional reset: copy the backup back over the workbook. This removes every record added after the backup."""
    if not BACKUP_PATH.exists():
        print("No backup exists yet, so this lab has not changed the workbook.")
        return
    temp_path = WORKBOOK_PATH.with_name("employee_nominations_restore_temp.xlsx")
    shutil.copy2(BACKUP_PATH, temp_path)
    os.replace(temp_path, WORKBOOK_PATH)
    print("Workbook restored to the original sample records.")


# Demonstrate add_nomination on its own
demo_nomination = Nomination(
    employee_id="EMP1015", employee_name="Ananya Rao", department="Marketing",
    designation="Brand Manager", award_category="Innovation Award",
    nomination_reason="Introduced a bilingual campaign that increased regional engagement.",
    nominated_by="Anjali Gupta, Marketing Director",
)
new_id = add_nomination(demo_nomination)
print("Saved", new_id, "with status Pending.")
print("Records in workbook now:", len(read_nominations()))
read_nominations().tail(1)

## SECTION 6 & 7 – Custom Tools: Search by Employee ID and by Employee Name

### How a Python function becomes a LangChain tool

A LangChain tool is a Python function that the LLM is allowed to call. The `@tool` decorator gives the function:

- **a name** (the function name), which the LLM uses to select it;
- **a description** (the docstring), which tells the LLM *when* to use it. This matters most, because the LLM chooses tools from their descriptions;
- **typed inputs** (the function arguments). Each argument can carry a description using `Annotated[..., Field(description=...)]`, so the LLM knows what to pass;
- **an output** (the returned text). The agent reads this text and writes the answer.

The tools below return readable text. They call the Section 5 functions, so the business logic is not repeated.

### Tool 1 – `search_nomination_by_employee_id`

Input: an employee ID. Output: every matching record, or a clear message when there is none.

### Tool 2 – `search_nomination_by_employee_name`

Input: a full or partial name. Output: matching records. Names are not unique, so when **more than one employee** matches, the tool lists each employee with their ID and tells the agent to ask the user which one is meant. Similar names are never treated as the same person.

In [ ]:
@tool
def search_nomination_by_employee_id(
    employee_id: Annotated[str, Field(description="Exact employee ID, for example EMP1001")],
) -> str:
    """Find all nomination records for one employee using the employee ID.
    Use this when the user gives an employee ID such as EMP1001."""
    if not employee_id.strip():
        return "Please give an employee ID, for example EMP1001."
    matches = search_by_employee_id(employee_id)
    if matches.empty:
        return f"No nomination records found for employee ID {employee_id.strip().upper()}."
    return f"Found {len(matches)} nomination record(s) for {employee_id.strip().upper()}:\n" + format_records(matches)


# Test the tool the way the agent will call it
print("Tool name :", search_nomination_by_employee_id.name)
print("Arguments :", list(search_nomination_by_employee_id.args))
print()
print(search_nomination_by_employee_id.invoke({"employee_id": "EMP1001"}))
print()
print(search_nomination_by_employee_id.invoke({"employee_id": "EMP9999"}))

In [ ]:
@tool
def search_nomination_by_employee_name(
    employee_name: Annotated[str, Field(description="Full or partial employee name, for example Priya or Rahul Sharma")],
) -> str:
    """Search nomination records by employee name (case-insensitive, partial match allowed).
    If more than one employee matches, the result lists every employee so the user can say which one is meant.
    Use this when the user gives a name instead of an employee ID."""
    if not employee_name.strip():
        return "Please give an employee name to search for, for example Priya or Rahul Sharma."
    matches = search_by_employee_name(employee_name)
    if matches.empty:
        return f"No nomination records found for a name containing '{employee_name.strip()}'."
    employees = matches.drop_duplicates("Employee_ID")
    if len(employees) > 1:
        options = "\n".join(
            f"- {row['Employee_ID']}: {row['Employee_Name']}, {row['Designation']}, {row['Department']}"
            for _, row in employees.iterrows()
        )
        return (f"{len(employees)} different employees match '{employee_name.strip()}'. "
                "Do not assume which one is meant. Ask the user to identify the correct employee.\n"
                f"{options}\n\nMatching records:\n{format_records(matches)}")
    return f"Found {len(matches)} nomination record(s):\n" + format_records(matches)


# Successful lookup, then two different people with the same name, then no match
print(search_nomination_by_employee_name.invoke({"employee_name": "Priya Nair"}))
print("\n" + "=" * 60 + "\n")
print(search_nomination_by_employee_name.invoke({"employee_name": "rahul sharma"}))
print("\n" + "=" * 60 + "\n")
print(search_nomination_by_employee_name.invoke({"employee_name": "Zara Quinn"}))

## SECTION 8 – Custom Tool 3: Add an Employee Nomination

This tool writes to Excel, so it has strict rules. It runs these steps in order, and any step can stop the process:

1. **Validate.** The details go through the `Nomination` Pydantic model. Missing or invalid fields are returned to the agent, which asks the user to correct them.
2. **Check for potential duplicates.** An employee can receive several nominations, so an existing employee ID is **not** rejected. A potential duplicate needs the same employee, the same award category and a very similar reason. The existing record is shown, and the user decides.
3. **Preview.** The first call always uses `user_confirmed=False`. It returns a summary and writes nothing. The agent shows the summary and asks the user to confirm.
4. **Guard the write.** Saving needs `user_confirmed=True` **and** the user's latest message must be a clear "yes". This is a code check, so the agent cannot skip confirmation by mistake.
5. **Save.** The record is added with status **Pending**, the workbook is saved safely, and the tool reports `SAVED` only after the write succeeds.

In [ ]:
LATEST_USER_MESSAGE = ""   # set by run_turn() before each agent turn (Section 11)
CONFIRMATION_PATTERN = re.compile(r"^\s*(yes|y|yeah|yep|confirm|confirmed|ok|okay|sure|go ahead|proceed|save)\b", re.IGNORECASE)
REFUSAL_PATTERN = re.compile(r"\b(no|not|don't|cancel|change|wait|stop)\b", re.IGNORECASE)


def user_has_confirmed(message: str) -> bool:
    """True only when the latest message starts with a clear 'yes' and has no 'no' or change request."""
    return bool(CONFIRMATION_PATTERN.search(message)) and not REFUSAL_PATTERN.search(message)


def format_preview(nomination: Nomination) -> str:
    """The summary the user reviews before confirming."""
    return (f"Employee ID: {nomination.employee_id}\n"
            f"Employee name: {nomination.employee_name}\n"
            f"Department: {nomination.department}\n"
            f"Designation: {nomination.designation}\n"
            f"Award category: {nomination.award_category}\n"
            f"Nomination reason: {nomination.nomination_reason}\n"
            f"Nominated by: {nomination.nominated_by}\n"
            f"Nomination date: {nomination.nomination_date}")


@tool
def add_employee_nomination(
    employee_id: Annotated[str, Field(description="Employee ID, format EMP + 4 digits, for example EMP1001")] = "",
    employee_name: Annotated[str, Field(description="Employee's full name")] = "",
    department: Annotated[str, Field(description="Employee's department, for example IT Operations")] = "",
    designation: Annotated[str, Field(description="Employee's job title, for example Senior Systems Engineer")] = "",
    award_category: Annotated[str, Field(description="Award category, for example Star Performer Award")] = "",
    nomination_reason: Annotated[str, Field(description="Why the employee is nominated (at least 10 characters)")] = "",
    nominated_by: Annotated[str, Field(description="Name and role of the nominating manager")] = "",
    nomination_date: Annotated[str, Field(description="Date as YYYY-MM-DD. Leave empty for today's date")] = "",
    user_confirmed: Annotated[bool, Field(description="False (default) = validate and preview only. True = save. Use True ONLY after the user explicitly confirmed the preview in their latest message.")] = False,
    allow_duplicate: Annotated[bool, Field(description="True only if the user has said they still want to add a potential duplicate")] = False,
) -> str:
    """Validate a new employee nomination and save it to the Excel workbook as Pending, but only after explicit user confirmation.
    Always call this first with user_confirmed=False to validate and get a preview. Show the preview and ask the user to confirm.
    Call again with user_confirmed=True only after the user has said yes to that preview."""
    details = {
        "employee_id": employee_id, "employee_name": employee_name, "department": department,
        "designation": designation, "award_category": award_category,
        "nomination_reason": nomination_reason, "nominated_by": nominated_by,
        "nomination_date": nomination_date,
    }
    # Empty fields are left out, so Pydantic reports them as "Field required"
    try:
        nomination = Nomination(**{key: value for key, value in details.items() if value not in (None, "")})
    except ValidationError as error:
        return "VALIDATION ERROR - NOT SAVED. Ask the user to correct these fields:\n" + explain_validation_error(error)

    duplicates = find_potential_duplicates(nomination)
    if not duplicates.empty and not allow_duplicate:
        return ("POTENTIAL DUPLICATE - NOT SAVED. A similar nomination already exists:\n" + format_records(duplicates)
                + "\nShow this record to the user and ask whether they still want to add the new nomination.")

    preview = format_preview(nomination)
    existing = len(search_by_employee_id(nomination.employee_id))
    note = (f"\nNote: {nomination.employee_id} already has {existing} nomination(s). "
            "A new nomination for the same employee is allowed.") if existing else ""

    if not user_confirmed:
        return f"PREVIEW - NOT SAVED. Show this summary to the user and ask them to confirm.\n{preview}{note}"
    if not user_has_confirmed(LATEST_USER_MESSAGE):
        return ("CONFIRMATION MISSING - NOT SAVED. The user has not explicitly confirmed in their latest message. "
                f"Show the preview and ask them to confirm.\n{preview}")

    try:
        nomination_id = add_nomination(nomination)
    except RuntimeError as error:
        return f"SAVE FAILED - NOT SAVED. {error}"
    return f"SAVED: Nomination {nomination_id} added to the Excel workbook with status Pending.\n{preview}"

In [ ]:
# Test the tool. None of these four calls writes to the workbook.
records_before = len(read_nominations())
valid_details = dict(
    employee_id="EMP1001", employee_name="Rahul Sharma", department="IT Operations",
    designation="Senior Systems Engineer", award_category="Star Performer Award",
    nominated_by="Neha Kapoor, Operations Manager",
)

print("A. Invalid reason (too short):")
print(add_employee_nomination.invoke({**valid_details, "nomination_reason": "Good work"}))

print("\nB. New reason, user not yet confirmed (preview only):")
print(add_employee_nomination.invoke({**valid_details,
      "nomination_reason": "He resolved a critical outage and reduced downtime for every service team."}))

print("\nC. Same reason as NOM-0001 (potential duplicate):")
print(add_employee_nomination.invoke({**valid_details,
      "nomination_reason": "Led the response to a critical network outage and cut service downtime by 40 percent."}))

print("\nD. Confirmation flag set, but the latest message is not a confirmation:")
LATEST_USER_MESSAGE = "Neha Kapoor, Operations Manager."
print(add_employee_nomination.invoke({**valid_details,
      "nomination_reason": "He resolved a critical outage and reduced downtime for every service team.",
      "user_confirmed": True}))

print(f"\nWorkbook records before: {records_before} | after: {len(read_nominations())}  (no write happened)")

## SECTION 9 – Custom Tool 4: Nomination Summary

Summary questions ("how many are pending?", "which department has the most nominations?") are answered by counting rows in Excel, not by the LLM guessing. The tool supports four views: `overview` (total and status counts), `status`, `department` and `award_category`. Counts are sorted from highest to lowest, so the top entry answers "which has the most".

## SECTION 10 – Register Custom Tools with LangChain

Registering a tool means putting it in the list the agent receives. For each tool, the agent sees:

- **Name**: the identifier the LLM uses to select the tool.
- **Description**: the docstring. Clear wording helps the LLM choose correctly, so each description says when to use the tool.
- **Arguments**: the typed inputs and their descriptions, sent to the LLM as a JSON schema.
- **Output**: the returned text, which the agent reads before it writes the answer.

We use the simplest supported approach: the `@tool` decorator and one list. There are no custom tool classes.

In [ ]:
@tool
def get_nomination_summary(
    group_by: Annotated[str, Field(description="One of: overview, status, department, award_category")] = "overview",
) -> str:
    """Count nominations in the Excel workbook. Use this for summary questions, such as how many nominations
    exist, how many are pending, approved or rejected, or which department or award category has the most nominations.
    group_by: 'overview' (total and status counts), 'status', 'department' or 'award_category'."""
    key = group_by.strip().lower()
    try:
        counts = build_nomination_summary(key)
    except ValueError as error:
        return str(error)
    titles = {"overview": "Nomination overview", "status": "Nominations by status",
              "department": "Nominations by department", "award_category": "Nominations by award category"}
    return titles.get(key, "Nominations") + ":\n" + "\n".join(f"- {name}: {count}" for name, count in counts.items())


for group in ["overview", "department"]:
    print(f"--- get_nomination_summary(group_by='{group}') ---")
    print(get_nomination_summary.invoke({"group_by": group}))
    print()

# Section 10: the four custom tools registered for the agent
ALL_TOOLS = [
    search_nomination_by_employee_id,
    search_nomination_by_employee_name,
    add_employee_nomination,
    get_nomination_summary,
]

for registered in ALL_TOOLS:
    print(f"{registered.name:34} args: {', '.join(registered.args)}")
    print(f"{'':34} description: {registered.description.splitlines()[0]}")

## SECTION 11 – Build the Nomination Management Agent

The agent has four parts:

1. **The LLM** (`ChatOpenAI`, `gpt-4o-mini`): reads the request and decides which tool to call.
2. **The tools** (`ALL_TOOLS`): the four custom functions from Sections 6 to 10.
3. **The prompt**: the assistant's role and the rules for writing. It explains what each tool is for, but it does **not** hard-code a tool for each request. The LLM chooses tools from the user's intent.
4. **The executor** (`AgentExecutor`): runs the loop. It sends the request to the LLM, runs any tool the LLM asks for, returns the result, and stops at a final answer or after `MAX_ITERATIONS` steps.

`verbose=True` prints every step, so you can see which tool was chosen and what it was given. The agent API matches Module 8 Lab 1: `create_tool_calling_agent` and `AgentExecutor` from `langchain_classic`.

In [ ]:
SYSTEM_PROMPT = """You are an Employee Nomination Management Assistant. Help users search existing nominations,
collect new nomination information, validate details and update the Excel workbook only after explicit user confirmation.

Rules:
1. Choose the tool that matches the user's intent. Each tool's description explains when to use it.
2. Answer questions about nominations only from tool results or earlier answers in this conversation.
   If a tool finds nothing, say so clearly. Never invent names, IDs, dates or statuses.
3. For a new nomination, collect: employee ID, employee name, department, designation, award category,
   nomination reason and nominated by. Ask only for the fields that are still missing. Do not invent employee details.
4. When all fields are present, call add_employee_nomination with user_confirmed=False. It validates the details and
   returns a preview. Show the preview and ask the user to confirm.
5. Call add_employee_nomination with user_confirmed=True ONLY after the user explicitly confirms in their latest message.
   Never set user_confirmed to True on your own.
6. If the tool reports a validation error, ask the user to correct the listed fields.
   If it reports a potential duplicate, show the existing record and ask whether to add anyway.
   Use allow_duplicate=True only if the user says yes to that question.
7. Say that a nomination was saved ONLY if the tool result starts with SAVED. Never claim a save otherwise.
   New nominations always have status Pending.
8. If a name matches more than one employee, list the matches with their IDs and ask which employee the user means.
9. Keep answers short and clear."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),       # earlier turns, so the agent remembers the conversation
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),   # tool calls and results from the current turn
])

In [ ]:
if not API_KEY_FOUND:
    raise EnvironmentError("OPENAI_API_KEY is missing. Add it to a .env file (see the README) and run this cell again.")

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)   # temperature 0 keeps answers consistent
agent = create_tool_calling_agent(llm, ALL_TOOLS, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=ALL_TOOLS,
    verbose=True,                    # print every step for the classroom
    max_iterations=MAX_ITERATIONS,   # safety limit on the loop
    return_intermediate_steps=True,  # keep the tool trace so we can report which tools ran
    handle_parsing_errors=True,
)

chat_history = []   # the conversation so far (human and AI messages)
SAVED_IDS = []      # Nomination IDs confirmed by a SAVED result from the add tool


def run_turn(user_text: str) -> str:
    """Send one user message to the agent, keep the conversation, and print the tools that were used."""
    global LATEST_USER_MESSAGE
    LATEST_USER_MESSAGE = user_text
    result = agent_executor.invoke({"input": user_text, "chat_history": chat_history})
    chat_history.append(HumanMessage(content=user_text))
    chat_history.append(AIMessage(content=result["output"]))
    steps = result["intermediate_steps"]
    for action, observation in steps:
        if str(observation).startswith("SAVED"):
            SAVED_IDS.append(re.search(r"NOM-\d{4}", str(observation)).group(0))
    print("\n=== TOOLS USED (in order) ===")
    print([action.tool for action, _ in steps] or "No tools were used.")
    print("\n=== AGENT RESPONSE ===")
    print(result["output"])
    return result["output"]


print("Agent ready. Model:", CHAT_MODEL, "| max steps:", MAX_ITERATIONS)

## SECTION 12 – Conversational Nomination Collection

The agent collects nomination details the way a colleague would: one question at a time, and only for what is still missing.

- **Context is kept.** Each turn sends the earlier messages along with the new one (`chat_history`), so the agent does not ask again for details you already gave.
- **Only missing required fields are requested.** Required: employee ID, name, department, designation, award category, reason and nominated by. The nomination date defaults to today.
- **No invented data.** If a field is missing, the agent asks for it.
- **Invalid or incomplete records are never saved.** Pydantic checks them first.
- **Explicit confirmation before writing.** The agent shows a summary and waits for a clear "yes".
- **No false success.** The agent reports a save only when the tool returns `SAVED`.

**The conversation loop.** The loop below is the classroom version: type a message, read the reply, and type `exit` to finish. Set `INTERACTIVE_CHAT = True` and run the cell to chat yourself. The scripted demonstrations in Section 13 use the same `run_turn()` function, so they behave like a live chat.

In [ ]:
def run_scripted_conversation(title: str, user_messages: list) -> None:
    """Replay fixed user messages through the same agent, so each demonstration can be repeated."""
    print("#" * 72)
    print(title)
    print("#" * 72)
    chat_history.clear()   # each demonstration starts a fresh conversation
    for message in user_messages:
        print(f"\nUSER: {message}")
        run_turn(message)


INTERACTIVE_CHAT = False   # set to True and run this cell to chat with the agent yourself


def interactive_chat() -> None:
    """Simple classroom loop. Type 'exit' (or 'quit') to stop."""
    chat_history.clear()
    print("Nomination assistant ready. Type 'exit' to stop.\n")
    while True:
        user_text = input("You: ").strip()
        if user_text.lower() in {"exit", "quit"}:
            print("Session ended.")
            break
        if user_text:
            print()
            run_turn(user_text)


if INTERACTIVE_CHAT:
    interactive_chat()
else:
    print("Interactive chat is off (INTERACTIVE_CHAT = False). The scripted demonstrations follow in Section 13.")

## SECTION 13 – Practical Demonstrations

Each use case below runs as a scripted conversation through the real agent, so it calls the real tools and the real Excel file. Watch the `TOOLS USED` line after each reply.

| Use case | What to look for |
|---|---|
| 1. New nomination | The agent asks only for missing fields, previews the summary, and saves only after "Yes, confirm." |
| 1b. Declined nomination | The agent previews, the user says no, and the workbook does not change. |
| 2. Search by employee ID | The agent calls the ID search tool and summarises the records. |
| 3. Search by name | "Rahul Sharma" matches two different employees, so the agent asks which one is meant. The follow-up uses the conversation context. |
| 4. Summary | The agent calls the summary tool and answers from the counts. |

**Run the notebook once on a fresh workbook.** After Use Case 1 saves Rahul Sharma's nomination, a second run finds that record and reports a potential duplicate. The agent then asks before adding anything. To start again, run `restore_sample_workbook()` in a new cell. It restores the original 14 records from the backup.

In [ ]:
records_before_uc1 = len(read_nominations())
run_scripted_conversation(
    "USE CASE 1 - New nomination: collect, validate, confirm, save",
    [
        "I want to nominate Rahul Sharma for the Star Performer Award.",
        "Rahul Sharma, employee ID EMP1001.",
        "IT Operations, Senior Systems Engineer.",
        "He successfully resolved a critical infrastructure outage and reduced service downtime.",
        "Neha Kapoor, Operations Manager.",
        "Yes, confirm.",
    ],
)
print(f"\nWorkbook records before: {records_before_uc1} | after: {len(read_nominations())}")

In [ ]:
records_before_decline = len(read_nominations())
run_scripted_conversation(
    "USE CASE 1b - User declines: nothing is written",
    [
        "Nominate EMP1002 Priya Nair for the Innovation Award. Department: Finance. Designation: Financial Analyst. "
        "Reason: She built a new forecasting template that saves the team time every month. "
        "Nominated by Suresh Menon, Finance Director.",
        "No, do not save this. Cancel it.",
    ],
)
print(f"\nWorkbook records before: {records_before_decline} | after: {len(read_nominations())}  (must be equal)")

In [ ]:
run_scripted_conversation(
    "USE CASE 2 - Search by employee ID",
    ["Find all nominations for employee EMP1001."],
)

run_scripted_conversation(
    "USE CASE 3 - Search by employee name (two different employees share this name)",
    [
        "Has Rahul Sharma been nominated previously?",
        "Only the one in IT Operations, please.",
    ],
)

run_scripted_conversation(
    "USE CASE 4 - Nomination summary",
    ["How many nominations are pending, and which department has the highest number of nominations?"],
)

## SECTION 14 – Excel Validation and Data Integrity

After a write, the notebook reloads the workbook from disk and checks what was actually saved, not what the agent said. The integrity rules in this lab are:

- The original worksheet and its columns are kept.
- Existing records are never deleted or changed. New records are appended at the end.
- A backup (`employee_nominations.xlsx.bak`) is created before the first write.
- A save is reported only after the reloaded file passes every check.
- If the workbook is open in Excel, the write stops and the tool reports `SAVE FAILED`.

In [ ]:
reloaded = read_nominations()   # read the workbook again from disk, not from memory
print("Records in workbook now:", len(reloaded))
print("Sheets in workbook     :", load_workbook(WORKBOOK_PATH).sheetnames)
print("\nLast five records:")
print(reloaded.tail(5).to_string(index=False))

if not SAVED_IDS:
    print("\nNo SAVED result in this session yet. Run Use Case 1 first, then run this cell again.")
else:
    new_id = SAVED_IDS[-1]
    new_row = reloaded[reloaded["Nomination_ID"] == new_id]
    required = ["Employee_ID", "Employee_Name", "Department", "Designation", "Award_Category",
                "Nomination_Reason", "Nominated_By", "Nomination_Date", "Status"]
    checks = {
        "New Nomination_ID exists exactly once": len(new_row) == 1,
        "Initial status is Pending": len(new_row) == 1 and new_row["Status"].iloc[0] == "Pending",
        "All required fields are filled": len(new_row) == 1 and bool(new_row[required].notna().values.all()),
        "Columns are unchanged": list(reloaded.columns) == COLUMNS,
        "Original records are preserved, in order": reloaded["Nomination_ID"].tolist()[:len(ORIGINAL_IDS)] == ORIGINAL_IDS,
    }
    print(f"\nIntegrity checks for {new_id}:")
    for label, passed in checks.items():
        print(("PASS  " if passed else "FAIL  ") + label)
    assert all(checks.values()), "One or more integrity checks failed"

## SECTION 15 – Hands-On Exercises and Quiz

### Exercises

1. **Search by employee ID.** Look up `EMP1011` with `search_by_employee_id()`, then ask the agent the same question. Which records come back, and what is the status of each?
2. **Search by employee name.** Ask the agent for "Michael" and then for "Rahul". Explain why the first search needed no clarification and the second did.
3. **Conversational nomination.** Set `INTERACTIVE_CHAT = True`, then nominate `EMP1013 Thomas Mathew` for the *Team Collaboration Award*. Leave out the nominator at first and see whether the agent asks for it. At the confirmation step, answer "No" and check that the workbook did not change.
4. **Add an optional field.** Add `employee_location: Optional[str] = None` to the `Nomination` model and to the tool arguments. Test with and without a location. Explain why the workbook would also need a new column before the value could be saved.
5. **Statistics by department and award category.** Ask the agent: "How many nominations are there in each department, and which award category is used most?" Compare its answer with `build_nomination_summary('department')` and `build_nomination_summary('award_category')`.

### Quiz (answers are in the instructor section at the end)

**Q1. Custom tools.** Which decorator turns a Python function into a LangChain tool that the agent can call?
A) `@property` B) `@tool` C) `@dataclass` D) `@staticmethod`

**Q2. Pydantic validation.** A user enters `"   "` (spaces only) as the employee ID. What happens?
A) It is saved as typed B) It is rejected because it is blank after stripping spaces C) It is replaced with `EMP0000` D) It is saved as Pending

**Q3. Employee lookup.** A user searches for "rahul" and two different employee IDs match. What should the agent do?
A) Show only the first match B) Merge both employees' records C) List both employees and ask which one is meant D) Refuse to search

**Q4. Tool calling.** What does the LLM use to decide which tool to call?
A) The tool's name, description and argument descriptions B) The row order in Excel C) A hard-coded `if/else` in the notebook D) The notebook's file name

**Q5. Conversation and Excel updates.** When may the add-nomination tool write to Excel?
A) As soon as the user mentions a nomination B) Only after the user explicitly confirms the preview C) Every time a search runs D) At the end of the notebook

---

## Instructor Section – Answers and Explanations

*Instructor use only.*

**Q1 – B (`@tool`).** The decorator builds the tool from the function name, the docstring and the typed arguments. The LLM sees these, so the docstring should say when to use the tool.

**Q2 – B.** Pydantic's `str_strip_whitespace` removes the spaces, leaving an empty value that fails `min_length=1`. The tool returns a validation error, the agent asks for a correct ID, and nothing is written.

**Q3 – C.** Names are not unique, so the search never assumes identity. The tool lists each employee with their ID, and the agent asks the user to choose. This is the behaviour shown in Use Case 3.

**Q4 – A.** The LLM never sees the Python code. It sees only the tool names, descriptions and argument schema, so clear wording directly affects which tool it chooses.

**Q5 – B.** The first call uses `user_confirmed=False` and only previews. The write also needs a code check on the user's latest message. This blocks a write without explicit confirmation, even if the LLM makes a mistake.